# I. 독립변수 16개(t-test + logistic regression) QSVM-KTA(커널-타깃 정렬) 버전

## 1. 라이브러리 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 데이터 불러오기 및 탐색 함수

In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/동국대학원/연구 과제/양자컴퓨팅 기반 머신러닝 하이브리드 모델을 활용한 기업신용평가모형/data/FILE2_중공업.xls'
df = pd.read_excel(path)
df.tail(5)

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 1. 설정
# ============================================================


# 결과 파일 저장 여부
SAVE_RESULTS = False
OUTPUT_DIR = "eda_results"

# 고상관 변수 기준
CORR_THRESHOLD = 0.95

ID = "회사코드"
INDUSTRY = "산업코드"
TARGET = "정상/폐업"
DATE = "재무비율기준/결산기준일"
AUDIT = "감사의견코드"

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


# ============================================================
# 2. PDF 참고: 산업코드 사전
# ============================================================

INDUSTRIES = {
    "D21": "펄프, 종이 및 종이제품 제조업",
    "D23": "코크스, 석유정제품 및 핵연료 제조업",
    "D24": "화합물 및 화학제품 제조업",
    "D25": "고무 및 플라스틱제품 제조업",
    "D26": "비금속 광물제품 제조업",
    "D27": "제1차 금속산업",
    "D28": "조립금속제품 제조업: 기계 및 가구 제외",
    "D29": "기타 기계 및 장비 제조업",
    "D30": "컴퓨터 및 사무용 기기제조업",
    "D31": "기타 전기기계 및 전기변환장치 제조업",
    "D32": "전자부품, 영상, 음향 및 통신장비 제조업",
    "D33": "의료, 정밀, 공학기기 및 시계 제조업",
    "D34": "자동차 및 트레일러 제조업",
    "D35": "기타 운송장비 제조업",
}


# ============================================================
# 3. PDF 참고: 변수 분류
#
# 변수명으로 직접 매칭한다.
# 비율, 금액, 배수, 기간 등이 섞여 있으므로
# 모든 수치 변수를 같은 단위라고 가정하면 안 된다.
# ============================================================

GROUP_NAMES = {
    "성장성": """
        매출액증가율|영업이익증가율|순이익증가율|총자산증가율|
        자기자본증가율|법인세차감전순이익증가율|유동자산증가율|
        유형자산증가율|재고자산증가율|종업원수증가율
    """,

    "수익성": """
        매출원가율|판관비율|영업이익률|법인세차감전순이익률|
        당기순이익율|ROIC|ROE|EBITDA마진율|재료비/매출액|
        노무비/매출액|경비/매출액|총자산법인세차감전순이익률|
        순금융비용부담율|자본금법인세차감전순이익률|감가상각률|
        감가상각비대총비용비율|법인세차감전순이익이자보상율|
        금융비용대총부채비율|금융비용대총비용비율|매출총이익률|
        배당성향|배당율|사내유보대자기자본비율|사내유보율|
        순외환손익대매출액|연구개발비대매출액|인건비대총비용비율|
        자기자본법인세차감전순이익률|자기자본배당률|자본금순이익률|
        조세공과대총비용비율|조세대조세차감전순이익률|차입금평균이자율|
        총자산순이익률(ROA)|총자산영업이익률|EBITDA(천원)|
        영업외수지율|경영자본영업이익율|수지비율|
        현금영업이익/매출액|현금흐름/매출액|FCF/매출액
    """,

    "안정성": """
        이자보상배수(배)|EBITDA/금융비용(배)|EBITDA/총차입금|
        차입금의존도|단기성차입금/총차입금|고정장기적합율|
        금융비용부담율|부채비율|자기자본비율|차입금/매출액|
        유동비율|EBITDA/단기차입금|차입금/자기자본비율|
        장단기대여금/총자산|적립금비율|비유동부채대순운전자본비율|
        비유동부채비율|비유동비율|당좌비율|매입채무대재고자산비율|
        매출채권대매입채무비율|매출채권대상/제품비율|법인세부담률|
        순운전자본대총자본비율|유동부채비율|유동자산구성비율|
        유보액대납입자본배율|유보액대총자산비율|
        재고자산대순운전자본비율|재고자산대유동자산비율|
        재무레버리지|차입금대월매출액|총현금흐름대당기상환부채|
        총현금흐름대매출액비율|총현금흐름대부채비율|
        총현금흐름대유동부채|총현금흐름대차입금비율|
        총현금흐름대총자본비율|비유동자산비율|단기부채상환능력|
        부채상환계수|총자산부채비율|적립금총자산비율|
        적립금비율2|현금비율|현금영업이익/금융비용|
        현금영업이익/단기차입금|현금영업이익/총차입금|
        현금흐름/총차입금|영업활동조달현금/금융비용|
        영업활동조달현금/차입금|FCF/차입금
    """,

    "활동성": """
        매출채권회전율(회)|재고자산회전율(회)|순운전자본회전율(회)|
        총자본회전율(회)|회전기간|비유동자산회전율(회)|
        매입채무회전기간|매입채무회전율(회)|매출채권회전기간|
        상(제)품회전율(회)|순영업자본회전율(회)|영업자본회전율(회)|
        순운전자본회전기간|원부재료회전율(회)|유동자산회전율(회)|
        유형자산회전율(회)|자기자본회전율(회)|자본금회전율(회)|
        재고자산회전기간|재공품회전율(회)|투하자산회전율(회)
    """,

    "생산성": """
        기계투자효율|노동소득분배율|부가가치(천원)|부가가치율|
        설비투자효율|이윤분배율|종업원1인당법인세차감전순이익|
        종업원1인당기계장비율|종업원1인당노동장비율|
        종업원1인당매출액|종업원1인당부가가치|종업원1인당순이익|
        종업원1인당영업이익|종업원1인당인건비|
        종업원1인당자본집약도|총자본투자효율|자본분배율
    """,

    "비재무/규모": """
        업력|경영자본|부가가치|종업원수|감사의견코드
    """,
}

GROUPS = {
    variable.strip(): group
    for group, variables in GROUP_NAMES.items()
    for variable in variables.split("|")
    if variable.strip()
}


# ============================================================
# 4. 데이터 읽기
# ============================================================

def read_data(path):
    """
    실제 파일 형식에 따라 xls/xlsx 엔진을 선택한다.

    회사코드는 식별자이므로 문자열로 읽는다.
    원본 파일 자체는 수정하지 않는다.
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"파일이 없습니다: {path.resolve()}")

    with path.open("rb") as f:
        signature = f.read(4)

    engine = "openpyxl" if signature == b"PK\x03\x04" else "xlrd"

    df = pd.read_excel(
        path,
        sheet_name=0,
        engine=engine,
        dtype={
            ID: "string",
            INDUSTRY: "string",
            AUDIT: "string",
        },
    )

    df.columns = df.columns.astype(str).str.strip()

    if df.columns.duplicated().any():
        raise ValueError("중복된 열 이름이 있습니다.")

    required = {ID, INDUSTRY, TARGET, DATE, AUDIT}
    missing_columns = required - set(df.columns)

    if missing_columns:
        raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")

    # 문자열의 앞뒤 공백만 정리
    for col in [ID, INDUSTRY, AUDIT]:
        df[col] = df[col].str.strip().replace("", pd.NA)

    return df


# ============================================================
# 5. 전체 EDA
# ============================================================

def explore_data(df, corr_threshold=0.95):
    """
    원자료를 임의 대치하거나 이상값을 삭제하지 않는 EDA.

    반환값
    -------
    x       : 수치형 분석용 복사본
    dates   : 파싱한 결산일
    tables  : 탐색 결과 DataFrame 사전
    summary : 전체 요약 사전
    """

    if not 0 <= corr_threshold <= 1:
        raise ValueError("상관계수 기준은 0~1 사이여야 합니다.")

    excluded = {ID, INDUSTRY, TARGET, DATE, AUDIT}

    numeric_features = [
        col for col in df.columns
        if col not in excluded
    ]

    # 숫자 변환 실패를 별도로 집계
    converted = df[numeric_features].apply(
        pd.to_numeric,
        errors="coerce",
    )

    conversion_failures = (
        df[numeric_features].notna()
        & converted.isna()
    )

    infinite = pd.DataFrame(
        np.isinf(converted),
        index=df.index,
        columns=numeric_features,
    )

    # 분석 복사본에서만 무한대를 NaN으로 처리
    # 원본의 결측, 0, 음수, 큰 수는 임의 변경하지 않음
    x = converted.replace([np.inf, -np.inf], np.nan)

    # --------------------------------------------------------
    # 결산일
    # --------------------------------------------------------

    date_text = (
        df[DATE]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    dates = pd.to_datetime(
        date_text,
        format="%Y%m%d",
        errors="coerce",
    )

    year = dates.dt.year.astype("Int64").rename("결산연도")

    # 타깃 결측이 있더라도 교차표에서 제외되지 않게 표시
    labels = df[TARGET].astype("string").fillna("(결측)")
    industries = df[INDUSTRY].fillna("(결측)")
    audits = df[AUDIT].fillna("(결측)")

    # --------------------------------------------------------
    # A. 변수 프로파일
    # --------------------------------------------------------

    profile = pd.DataFrame({
        "변수": df.columns,
        "자료형": df.dtypes.astype(str).values,
        "분류": [
            GROUPS.get(col, "식별/분류/기준일 또는 미분류")
            for col in df.columns
        ],
        "결측수": df.isna().sum().values,
        "결측비율": df.isna().mean().values,
        "고유값수": df.nunique(dropna=True).values,
    })

    # --------------------------------------------------------
    # B. 수치형 기술통계
    # --------------------------------------------------------

    stats = x.describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ).T

    stats.index.name = "변수"

    valid_count = x.notna().sum().replace(0, np.nan)

    stats["결측비율_분석용"] = x.isna().mean()
    stats["0개수"] = x.eq(0).sum()
    stats["0비율_전체행기준"] = x.eq(0).mean()
    stats["0비율_유효값기준"] = x.eq(0).sum() / valid_count
    stats["음수개수"] = x.lt(0).sum()
    stats["음수비율_유효값기준"] = x.lt(0).sum() / valid_count
    stats["무한대개수"] = infinite.sum()
    stats["숫자변환실패수"] = conversion_failures.sum()
    stats["왜도"] = x.skew()
    stats["첨도"] = x.kurt()

    # --------------------------------------------------------
    # C. IQR 기반 극단값 탐색
    # --------------------------------------------------------

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_flags = (
        x.lt(lower)
        | x.gt(upper)
    ) & x.notna()

    stats["IQR하한"] = lower
    stats["IQR상한"] = upper
    stats["IQR극단값수"] = outlier_flags.sum()
    stats["IQR극단값비율"] = outlier_flags.sum() / valid_count
    stats["IQR이0인지"] = iqr.eq(0)

    # 주의:
    # IQR 기준을 벗어났다는 이유만으로 오류라고 판단하지 않음.
    # 분모가 작은 재무비율은 정상적인 값도 매우 커질 수 있음.
    # IQR=0인 변수에서는 특히 기계적인 해석을 피할 것.

    # --------------------------------------------------------
    # D. 완전히 동일한 변수
    # --------------------------------------------------------

    identical_pairs = []

    for i, col1 in enumerate(numeric_features):
        for col2 in numeric_features[i + 1:]:
            if df[col1].equals(df[col2]):
                identical_pairs.append((col1, col2))

    identical_table = pd.DataFrame(
        identical_pairs,
        columns=["변수1", "변수2"],
    )

    # --------------------------------------------------------
    # E. 상태 코드 빈도
    # --------------------------------------------------------

    target_counts = (
        labels
        .value_counts(dropna=False)
        .rename_axis(TARGET)
        .reset_index(name="행수")
    )

    target_counts["비율"] = target_counts["행수"] / len(df)

    # --------------------------------------------------------
    # F. 산업/연도/감사의견과 상태 코드
    # --------------------------------------------------------

    industry_target = pd.crosstab(
        industries,
        labels,
    )

    industry_target.index.name = INDUSTRY
    industry_target.columns.name = TARGET

    industry_target.insert(
        0,
        "산업명",
        [
            INDUSTRIES.get(code, "미확인")
            for code in industry_target.index
        ],
    )

    # -1은 결산일 파싱 실패 또는 결측
    year_target = pd.crosstab(
        year.fillna(-1),
        labels,
    )

    year_target.index.name = "결산연도"

    # 각 상태 코드 내부에서 연도가 차지하는 비율
    year_share_within_target = year_target.div(
        year_target.sum(axis=0).replace(0, np.nan),
        axis=1,
    )

    # 각 연도 내부에서 상태 코드가 차지하는 비율
    target_share_within_year = year_target.div(
        year_target.sum(axis=1).replace(0, np.nan),
        axis=0,
    )

    audit_target = pd.crosstab(
        audits,
        labels,
    )

    audit_target.index.name = AUDIT

    # --------------------------------------------------------
    # G. 상태 코드별 수치 통계와 결측률
    # --------------------------------------------------------

    grouped_stats = []
    grouped_missing = []

    for label in labels.unique():
        mask = labels.eq(label)
        block = x.loc[mask]

        part = (
            block
            .agg(["count", "mean", "median", "std"])
            .T
            .rename_axis("변수")
            .reset_index()
        )

        part.insert(0, TARGET, label)
        grouped_stats.append(part)

        missing_part = (
            block.isna()
            .mean()
            .rename_axis("변수")
            .reset_index(name="결측비율")
        )

        missing_part.insert(0, TARGET, label)
        grouped_missing.append(missing_part)

    stats_by_target = pd.concat(
        grouped_stats,
        ignore_index=True,
    )

    missing_by_target = pd.concat(
        grouped_missing,
        ignore_index=True,
    )

    # --------------------------------------------------------
    # H. 상관관계
    # --------------------------------------------------------

    correlation_tables = {}

    for method in ["pearson", "spearman"]:
        corr = x.corr(
            method=method,
            min_periods=30,
        )

        pairs = []

        for i, col1 in enumerate(numeric_features):
            for col2 in numeric_features[i + 1:]:
                value = corr.loc[col1, col2]

                if pd.notna(value) and abs(value) >= corr_threshold:
                    n_pairs = (
                        x[col1].notna()
                        & x[col2].notna()
                    ).sum()

                    pairs.append({
                        "변수1": col1,
                        "변수2": col2,
                        "상관계수": value,
                        "유효쌍수": int(n_pairs),
                    })

        high_corr = pd.DataFrame(
            pairs,
            columns=["변수1", "변수2", "상관계수", "유효쌍수"],
        )

        if not high_corr.empty:
            high_corr = high_corr.sort_values(
                "상관계수",
                key=lambda s: s.abs(),
                ascending=False,
            )

        correlation_tables[f"{method}_matrix"] = corr
        correlation_tables[f"{method}_highcorr"] = high_corr

    # --------------------------------------------------------
    # I. 기업별 행 수
    # --------------------------------------------------------

    company_counts = (
        df[ID]
        .value_counts(dropna=False)
        .rename_axis(ID)
        .reset_index(name="행수")
    )

    # --------------------------------------------------------
    # J. 산업 × 연도 × 상태
    # --------------------------------------------------------

    cross = pd.DataFrame({
        "결산연도": year,
        INDUSTRY: industries,
        TARGET: labels,
    })

    year_industry_target = (
        cross
        .groupby(
            ["결산연도", INDUSTRY, TARGET],
            dropna=False,
        )
        .size()
        .reset_index(name="행수")
    )

    # --------------------------------------------------------
    # K. 파싱 이상 행
    # --------------------------------------------------------

    parse_issues = df[[ID, DATE]].copy()

    # 첫 행이 헤더라고 가정한 원본 Excel 행 번호
    parse_issues.insert(
        0,
        "엑셀행",
        np.arange(len(df)) + 2,
    )

    parse_issues["결산일파싱실패"] = dates.isna()
    parse_issues["숫자변환실패수"] = conversion_failures.sum(axis=1)
    parse_issues["무한대수"] = infinite.sum(axis=1)

    issue_mask = (
        parse_issues["결산일파싱실패"]
        | parse_issues["숫자변환실패수"].gt(0)
        | parse_issues["무한대수"].gt(0)
    )

    parse_issues = parse_issues.loc[issue_mask]

    # --------------------------------------------------------
    # L. 변수 사전
    # --------------------------------------------------------

    variable_dictionary = profile[["변수", "분류"]].copy()

    cautions = {
        TARGET: (
            "1/2의 의미와 상태 판정시점 확인 필요. "
            "현재 상태 분류와 미래 부도 예측을 구분할 것."
        ),
        DATE: (
            "예측시점 및 재무정보 이용 가능 시점 확인 필요. "
            "상태별 결산연도 분포 차이를 점검할 것."
        ),
        ID: "기업 식별자. 일반적으로 예측 독립변수에서 제외.",
        "업력": (
            "PDF K1 산식의 기업도산일 참조에 따른 "
            "예측시점 이후 정보 포함 가능성 확인 필요."
        ),
        "순운전자본회전기간": (
            "명칭과 PDF 산식 간 불일치 가능성 확인 필요."
        ),
        AUDIT: (
            "숫자형 연속변수가 아닌 범주형 변수. "
            "미확인 코드의 의미를 임의로 지정하지 말 것."
        ),
    }

    variable_dictionary["주의사항"] = (
        variable_dictionary["변수"]
        .map(cautions)
        .fillna("")
    )

    # --------------------------------------------------------
    # M. 결과 사전
    # --------------------------------------------------------

    tables = {
        "profile": profile,
        "numeric_stats": stats.reset_index(),
        "target_counts": target_counts,
        "industry_target": industry_target.reset_index(),
        "year_target": year_target.reset_index(),
        "year_share_within_target": (
            year_share_within_target.reset_index()
        ),
        "target_share_within_year": (
            target_share_within_year.reset_index()
        ),
        "audit_target": audit_target.reset_index(),
        "identical_columns": identical_table,
        "stats_by_target": stats_by_target,
        "missing_by_target": missing_by_target,
        "company_counts": company_counts,
        "year_industry_target": year_industry_target,
        "parse_issues": parse_issues,
        "variable_dictionary": variable_dictionary,
        "duplicate_rows": df.loc[df.duplicated(keep=False)],
    }

    tables.update(correlation_tables)

    summary = {
        "행수": len(df),
        "열수": len(df.columns),
        "고유기업수": int(df[ID].nunique()),
        "수치분석변수수": len(numeric_features),
        "전체결측셀수": int(df.isna().sum().sum()),
        "완전중복행수": int(df.duplicated().sum()),
        "회사_결산일중복수": int(
            df.duplicated([ID, DATE]).sum()
        ),
        "결산일최소": str(dates.min()),
        "결산일최대": str(dates.max()),
        "결산일파싱실패수": int(dates.isna().sum()),
        "숫자변환실패셀수": int(
            conversion_failures.sum().sum()
        ),
        "무한대셀수": int(infinite.sum().sum()),
        "전체결측변수": x.columns[
            x.notna().sum().eq(0)
        ].tolist(),
        "상수변수_결측제외": x.columns[
            x.nunique(dropna=True).eq(1)
        ].tolist(),
        "완전동일변수쌍": identical_pairs,
        "산업사전미등록코드": sorted(
            set(df[INDUSTRY].dropna()) - set(INDUSTRIES)
        ),
    }

    return x, dates, tables, summary


# ============================================================
# 6. 보기 좋게 출력하기
# ============================================================

def show_table(title, table, n=20):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    shown = table if n is None else table.head(n)

    try:
        from IPython.display import display
        display(shown)
    except ImportError:
        print(shown.to_string(index=False))


# ============================================================
# 7. 선택 사항: 탐색 결과 저장
# ============================================================

def save_results(tables, summary, output_dir):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # 상관행렬은 변수명이 인덱스에 있으므로 함께 저장
    matrix_names = {"pearson_matrix", "spearman_matrix"}

    with pd.ExcelWriter(
        out / "eda_tables.xlsx",
        engine="openpyxl",
    ) as writer:

        for name, table in tables.items():
            include_index = name in matrix_names

            table.to_excel(
                writer,
                sheet_name=name[:31],
                index=include_index,
            )

            table.to_csv(
                out / f"{name}.csv",
                index=include_index,
                encoding="utf-8-sig",
            )

            ws = writer.sheets[name[:31]]
            ws.freeze_panes = "B2" if include_index else "A2"
            ws.auto_filter.ref = ws.dimensions

    (out / "summary.json").write_text(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(f"\n결과 저장 위치: {out.resolve()}")

## 3. 전체 탐색 실행

In [ ]:
# ============================================================
# 데이터 로딩
# ============================================================

print("데이터 크기:", df.shape)

show_table(
    "원본 데이터 앞부분",
    df,
    n=5,
)


# ============================================================
# EDA 실행
# ============================================================

x, dates, tables, summary = explore_data(
    df,
    corr_threshold=CORR_THRESHOLD,
)

print("\n[전체 요약]")
print(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    )
)


# ============================================================
# 1. 변수 구성
# ============================================================

show_table(
    "변수별 자료형 / 분류 / 결측 / 고유값",
    tables["profile"],
    n=None,
)

show_table(
    "변수 분류별 개수",
    tables["profile"]
    .groupby("분류", dropna=False)
    .size()
    .reset_index(name="변수수"),
)


# ============================================================
# 2. 결측치
# ============================================================

show_table(
    "결측률 상위 변수",
    tables["profile"].sort_values(
        "결측비율",
        ascending=False,
    ),
    n=30,
)

# 행별 결측률도 확인
row_missing_rate = x.isna().mean(axis=1)

show_table(
    "기업별 수치변수 결측률 요약",
    row_missing_rate.describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
    .rename_axis("통계량")
    .reset_index(name="결측비율"),
)

show_table(
    "결측률이 높은 기업",
    df[[ID, INDUSTRY, TARGET, DATE]]
    .assign(수치변수결측비율=row_missing_rate)
    .sort_values(
        "수치변수결측비율",
        ascending=False,
    ),
    n=20,
)


# ============================================================
# 3. 수치 분포 / 0 / 음수 / 극단값
# ============================================================

show_table(
    "수치변수 기술통계",
    tables["numeric_stats"],
    n=20,
)

show_table(
    "0 비율이 높은 변수",
    tables["numeric_stats"].sort_values(
        "0비율_유효값기준",
        ascending=False,
    ),
    n=20,
)

show_table(
    "음수 비율이 높은 변수",
    tables["numeric_stats"].sort_values(
        "음수비율_유효값기준",
        ascending=False,
    ),
    n=20,
)

show_table(
    "IQR 기준 극단값 비율이 높은 변수 — 자동 삭제 대상은 아님",
    tables["numeric_stats"].sort_values(
        "IQR극단값비율",
        ascending=False,
    ),
    n=25,
)


# ============================================================
# 4. 타깃 분포
# ============================================================

show_table(
    "정상/폐업 코드별 행 수",
    tables["target_counts"],
)

show_table(
    "산업별 상태 코드 분포",
    tables["industry_target"],
    n=None,
)

show_table(
    "감사의견별 상태 코드 분포",
    tables["audit_target"],
    n=None,
)


# ============================================================
# 5. 시점 교란 점검
# ============================================================

show_table(
    "결산연도별 상태 코드 수",
    tables["year_target"],
    n=None,
)

show_table(
    "각 상태 코드 내부에서 연도별 구성비",
    tables["year_share_within_target"],
    n=None,
)

show_table(
    "각 연도 내부에서 상태 코드별 구성비",
    tables["target_share_within_year"],
    n=None,
)


# ============================================================
# 6. 상태 코드별 결측 차이
# ============================================================

missing_comparison = (
    tables["missing_by_target"]
    .pivot(
        index="변수",
        columns=TARGET,
        values="결측비율",
    )
)

# 여러 상태 코드 사이의 결측률 최대 차이
missing_comparison["결측률최대차이"] = (
    missing_comparison.max(axis=1)
    - missing_comparison.min(axis=1)
)

show_table(
    "상태 코드에 따라 결측률 차이가 큰 변수",
    missing_comparison
    .sort_values(
        "결측률최대차이",
        ascending=False,
    )
    .reset_index(),
    n=30,
)


# ============================================================
# 7. 주요 재무변수의 코드별 통계
# ============================================================

KEY_VARIABLES = [
    "영업이익률",
    "총자산순이익률(ROA)",
    "ROE",
    "부채비율",
    "자기자본비율",
    "유동비율",
    "차입금의존도",
    "이자보상배수(배)",
    "현금흐름/매출액",
    "종업원수",
    "업력",
]

show_table(
    "주요 변수의 상태 코드별 통계",
    tables["stats_by_target"]
    .loc[
        lambda t: t["변수"].isin(KEY_VARIABLES)
    ]
    .sort_values(["변수", TARGET]),
    n=None,
)

median_comparison = (
    tables["stats_by_target"]
    .pivot(
        index="변수",
        columns=TARGET,
        values="median",
    )
)

show_table(
    "주요 변수의 코드별 중앙값",
    median_comparison
    .reindex(
        [
            col for col in KEY_VARIABLES
            if col in median_comparison.index
        ]
    )
    .reset_index(),
    n=None,
)


# ============================================================
# 8. 중복 변수 / 다중공선성 후보
# ============================================================

show_table(
    "완전히 동일한 변수 쌍",
    tables["identical_columns"],
    n=None,
)

show_table(
    "Pearson 고상관 변수 쌍",
    tables["pearson_highcorr"],
    n=30,
)

show_table(
    "Spearman 고상관 변수 쌍",
    tables["spearman_highcorr"],
    n=30,
)


# ============================================================
# 9. 데이터 구조 / 파싱 오류
# ============================================================

show_table(
    "기업별 관측 행 수",
    tables["company_counts"].sort_values(
        "행수",
        ascending=False,
    ),
    n=10,
)

show_table(
    "파싱 이상 행",
    tables["parse_issues"],
    n=20,
)

show_table(
    "해석 시 주의사항이 있는 변수",
    tables["variable_dictionary"].loc[
        lambda t: t["주의사항"].ne("")
    ],
    n=None,
)


# ============================================================
# 10. 선택 저장
# ============================================================

if SAVE_RESULTS:
    save_results(
        tables,
        summary,
        OUTPUT_DIR,
    )

## 4. 전처리

### 기존 전처리

In [ ]:
# 처음 한 번만 실행
%pip install -U scikit-learn

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted


# ============================================================
# 1. 설정
# ============================================================

SEED = 42

ID_COL = "회사코드"
TARGET_COL = "정상/폐업"
DATE_COL = "재무비율기준/결산기준일"

CATEGORICAL_COLS = [
    "산업코드",
    "감사의견코드",
]

# 예측시점 이후 정보가 포함될 가능성이 있어 우선 제외
SUSPECTED_LEAKAGE_COLS = ["업력"]

# 훈련셋에서 결측률이 60%를 초과하는 수치변수 제거
MISSING_THRESHOLD = 0.60

# 훈련셋 기준 양쪽 1% 극단값 제한
LOWER_QUANTILE = 0.01
UPPER_QUANTILE = 0.99


# ============================================================
# 2. 훈련셋에서 수치변수 선택
#
# - 결측률 과다 변수
# - 상수/전체 결측 변수
# - 완전히 같은 수치변수
#
# 해당 판단은 fit()에 전달된 데이터에서만 수행한다.
# ============================================================

class NumericColumnSelector(BaseEstimator, TransformerMixin):

    def __init__(self, missing_threshold=0.60):
        self.missing_threshold = missing_threshold

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("입력은 pandas DataFrame이어야 합니다.")

        self.feature_names_in_ = np.asarray(X.columns, dtype=object)

        self.missing_rates_ = X.isna().mean()

        self.high_missing_cols_ = self.missing_rates_[
            self.missing_rates_ > self.missing_threshold
        ].index.tolist()

        self.constant_cols_ = X.columns[
            X.nunique(dropna=True) <= 1
        ].tolist()

        excluded = set(
            self.high_missing_cols_ + self.constant_cols_
        )

        candidates = [
            col for col in X.columns
            if col not in excluded
        ]

        self.duplicate_mapping_ = {}
        retained = []

        for col in candidates:
            duplicate_of = next(
                (
                    previous
                    for previous in retained
                    if X[col].equals(X[previous])
                ),
                None,
            )

            if duplicate_of is None:
                retained.append(col)
            else:
                self.duplicate_mapping_[col] = duplicate_of

        if not retained:
            raise ValueError("선택 후 남는 수치변수가 없습니다.")

        self.selected_columns_ = retained
        return self

    def transform(self, X):
        check_is_fitted(self, "selected_columns_")

        missing = set(self.selected_columns_) - set(X.columns)
        if missing:
            raise ValueError(f"필수 수치변수 누락: {missing}")

        return X.loc[:, self.selected_columns_].copy()

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "selected_columns_")
        return np.asarray(self.selected_columns_, dtype=object)


# ============================================================
# 3. 훈련셋 분위수 기반 극단값 제한
#
# 결측 여부가 보존되도록 대치 전에 실행한다.
# 행은 삭제하지 않는다.
# ============================================================

class QuantileClipper(BaseEstimator, TransformerMixin):

    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        if not 0 <= self.lower < self.upper <= 1:
            raise ValueError("0 <= lower < upper <= 1 이어야 합니다.")

        self.feature_names_in_ = np.asarray(X.columns, dtype=object)

        self.lower_bounds_ = X.quantile(self.lower)
        self.upper_bounds_ = X.quantile(self.upper)

        # 0이 많은 변수에서는 두 분위수가 같아질 수 있다.
        # 그때 전체 값을 같은 값으로 압축하지 않도록 제한을 생략한다.
        usable = self.upper_bounds_ > self.lower_bounds_

        self.lower_bounds_ = self.lower_bounds_.where(
            usable,
            -np.inf,
        )

        self.upper_bounds_ = self.upper_bounds_.where(
            usable,
            np.inf,
        )

        return self

    def transform(self, X):
        check_is_fitted(self, "lower_bounds_")

        return X.clip(
            lower=self.lower_bounds_,
            upper=self.upper_bounds_,
            axis=1,
        )

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "feature_names_in_")
        return self.feature_names_in_


# ============================================================
# 4. 수치 결측 여부 표시
#
# 훈련셋에서 결측이 없었어도 미래 데이터에는 생길 수 있으므로
# 선택된 수치변수 전체에 대해 표시 변수를 만든다.
# ============================================================

class MissingFlags(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        check_is_fitted(self, "feature_names_in_")
        return X.isna().astype(np.float64).to_numpy()

    def get_feature_names_out(self, input_features=None):
        return np.asarray(
            [f"{col}__missing" for col in self.feature_names_in_],
            dtype=object,
        )


# ============================================================
# 5. 입력 형태 정리
#
# 이 단계는 통계량을 학습하지 않는다.
# 추후 새로운 기업 데이터를 넣을 때도 같은 함수를 사용한다.
# ============================================================

def prepare_features(data):
    required = set(CATEGORICAL_COLS)

    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"범주형 변수 누락: {missing}")

    drop_cols = [
        ID_COL,
        TARGET_COL,
        DATE_COL,
        *SUSPECTED_LEAKAGE_COLS,
    ]

    X = data.drop(
        columns=drop_cols,
        errors="ignore",
    ).copy()

    # 범주형: 의미 미확인 코드는 그대로 유지한다.
    # DU, NS 등의 의미를 임의 추정하거나 순서를 부여하지 않음.
    for col in CATEGORICAL_COLS:
        values = X[col].astype("string").str.strip()
        values = values.mask(values.eq(""), pd.NA)

        # sklearn에서 일관되게 다루도록 np.nan으로 통일
        X[col] = values.astype(object)
        X.loc[values.isna(), col] = np.nan

    numeric_cols = [
        col for col in X.columns
        if col not in CATEGORICAL_COLS
    ]

    for col in numeric_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    X[numeric_cols] = X[numeric_cols].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    return X


# ============================================================
# 6. 타깃과 행 정리
# ============================================================

data = df.copy()
data.columns = data.columns.astype(str).str.strip()

required_columns = {
    ID_COL,
    TARGET_COL,
    DATE_COL,
    *CATEGORICAL_COLS,
}

missing_columns = required_columns - set(data.columns)
if missing_columns:
    raise ValueError(f"필수 컬럼 누락: {missing_columns}")

# 라벨 결측 행은 지도학습에서 제외
target_text = (
    data[TARGET_COL]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .replace("", pd.NA)
)

print("타깃 결측으로 제외하는 행 수:", target_text.isna().sum())

data = data.loc[target_text.notna()].copy()
target_text = target_text.loc[data.index]

# 완전히 동일한 행만 제거
duplicate_mask = data.duplicated()
print("완전 중복으로 제외하는 행 수:", duplicate_mask.sum())

data = data.loc[~duplicate_mask].copy()
target_text = target_text.loc[data.index]

# 기업이 여러 행이면 기업 단위 분할이 필요하므로
# 같은 기업이 train/test에 섞이지 않도록 여기서는 중단한다.
company_ids = data[ID_COL].astype("string").str.strip()

if company_ids.isna().any() or company_ids.eq("").any():
    raise ValueError("회사코드 결측을 먼저 확인해 주세요.")

if company_ids.duplicated().any():
    raise ValueError(
        "기업당 여러 행이 있습니다. "
        "현재 코드 대신 기업 단위 그룹 분할을 사용해야 합니다."
    )

observed_labels = set(target_text.unique())

if observed_labels != {"1", "2"}:
    raise ValueError(
        f"예상하지 못한 타깃 코드: {observed_labels}"
    )

# 의미를 추정하지 않고 단순히 이진 코드로 변환
# 0 = 원본 코드 1
# 1 = 원본 코드 2
# 양성 클래스 1이 '폐업'이라는 뜻은 아직 아니다.
TARGET_MAPPING = {"1": 0, "2": 1}

y = target_text.map(TARGET_MAPPING).astype("int64")
X = prepare_features(data)

# 행 인덱스 정렬
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

metadata = data[
    [ID_COL, DATE_COL]
].reset_index(drop=True)

metadata["결산연도"] = pd.to_datetime(
    metadata[DATE_COL]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True),
    format="%Y%m%d",
    errors="coerce",
).dt.year

numeric_cols = [
    col for col in X.columns
    if col not in CATEGORICAL_COLS
]

print("\n타깃 매핑:", TARGET_MAPPING)
print("전처리 전 X:", X.shape)
print("수치형 변수 수:", len(numeric_cols))
print("범주형 변수:", CATEGORICAL_COLS)


# ============================================================
# 7. 훈련 / 검증 / 테스트 분리
#
# 60% / 20% / 20%
# 이 분할은 동일 표본 내부의 비교용이다.
# 미래 시점 일반화 검증을 보장하지는 않는다.
# ============================================================

all_indices = np.arange(len(X))

train_idx, remaining_idx = train_test_split(
    all_indices,
    test_size=0.40,
    stratify=y,
    random_state=SEED,
)

valid_idx, test_idx = train_test_split(
    remaining_idx,
    test_size=0.50,
    stratify=y.iloc[remaining_idx],
    random_state=SEED,
)

X_train = X.iloc[train_idx].copy()
X_valid = X.iloc[valid_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_valid = y.iloc[valid_idx].copy()
y_test = y.iloc[test_idx].copy()

meta_train = metadata.iloc[train_idx].copy()
meta_valid = metadata.iloc[valid_idx].copy()
meta_test = metadata.iloc[test_idx].copy()


# ============================================================
# 8. 전처리 파이프라인
# ============================================================

# 수치 값:
# 극단값 제한 → 중앙값 대치 → RobustScaler
numeric_values_pipeline = Pipeline([
    (
        "clip",
        QuantileClipper(
            lower=LOWER_QUANTILE,
            upper=UPPER_QUANTILE,
        ),
    ),
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    (
        "scaler",
        RobustScaler(),
    ),
])

# 선택한 수치변수를 값과 결측 여부 두 갈래로 처리
numeric_branches = ColumnTransformer(
    transformers=[
        (
            "values",
            numeric_values_pipeline,
            slice(None),
        ),
        (
            "missing",
            MissingFlags(),
            slice(None),
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

numeric_pipeline = Pipeline([
    (
        "select",
        NumericColumnSelector(
            missing_threshold=MISSING_THRESHOLD,
        ),
    ),
    (
        "branches",
        numeric_branches,
    ),
])

# 산업코드와 감사의견에는 임의의 대소 관계를 부여하지 않는다.
# 결측은 별도 범주로 처리.
# 훈련 중 보지 못한 새 범주는 오류 대신 모두 0으로 처리된다.
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="__MISSING__",
            keep_empty_features=True,
        ),
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float64,
        ),
    ),
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_cols,
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_COLS,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)


# ============================================================
# 9. 훈련셋에만 fit
# ============================================================

X_train_array = preprocessor.fit_transform(X_train, y_train)

# 검증/테스트에서는 fit하지 않음!
X_valid_array = preprocessor.transform(X_valid)
X_test_array = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_array,
    index=X_train.index,
    columns=feature_names,
)

X_valid_processed = pd.DataFrame(
    X_valid_array,
    index=X_valid.index,
    columns=feature_names,
)

X_test_processed = pd.DataFrame(
    X_test_array,
    index=X_test.index,
    columns=feature_names,
)


# ============================================================
# 10. 결과 검증
# ============================================================

for name, processed in [
    ("train", X_train_processed),
    ("valid", X_valid_processed),
    ("test", X_test_processed),
]:
    assert np.isfinite(processed.to_numpy()).all(), (
        f"{name}에 결측 또는 무한대가 남았습니다."
    )

    assert processed.columns.equals(
        X_train_processed.columns
    ), f"{name}의 열 구성이 다릅니다."

assert set(meta_train[ID_COL]).isdisjoint(
    set(meta_valid[ID_COL])
)

assert set(meta_train[ID_COL]).isdisjoint(
    set(meta_test[ID_COL])
)

assert set(meta_valid[ID_COL]).isdisjoint(
    set(meta_test[ID_COL])
)

selector = (
    preprocessor
    .named_transformers_["numeric"]
    .named_steps["select"]
)

print("\n[전처리 완료]")
print("훈련:", X_train_processed.shape)
print("검증:", X_valid_processed.shape)
print("테스트:", X_test_processed.shape)

print("\n결측률 과다로 제거:")
print(selector.high_missing_cols_)

print("\n상수 또는 전체 결측으로 제거:")
print(selector.constant_cols_)

print("\n중복 제거 변수 → 남긴 변수:")
print(selector.duplicate_mapping_)

print("\n남긴 수치변수 수:")
print(len(selector.selected_columns_))

print("\n최종 독립변수 수:")
print(len(feature_names))

print("\n훈련 타깃 비율:")
print(y_train.value_counts(normalize=True).sort_index())

print("\n전처리 결과 앞부분:")
print(X_train_processed.iloc[:5, :12])

### 독립변수 16개 추출

In [ ]:
# ============================================================
# 148개 전처리 결과에서 16개 변수만 뽑아 df16 만들기
# (위 전처리 셀을 먼저 실행한 뒤 이 셀 실행)
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

FEATURES_16 = [
    '영업이익률', '재료비/매출액', '감가상각률', '금융비용대총부채비율',
    '금융비용대총비용비율', 'EBITDA/총차입금', '차입금의존도',
    '장단기대여금/총자산', '법인세부담률', '순운전자본대총자본비율',
    '유동자산구성비율', '유보액대총자산비율', '재고자산대유동자산비율',
    '매입채무회전기간', '자본금회전율(회)', '업력',
]
feat15 = [c for c in FEATURES_16 if c != '업력']   # 업력은 148개 전처리에서 빠져 있음

# 1) 15개: 처리된 열 이름에서 값 열(values__)만 정확히 찾기
def find_col(processed, name):
    hits = [c for c in processed.columns if c.endswith("values__" + name)]
    assert len(hits) == 1, (name, hits)
    return hits[0]

col_map = {name: find_col(X_train_processed, name) for name in feat15}

# 2) 업력(K1): 148개 버전에서 제외됐으므로 같은 방식으로 따로 처리 (훈련셋에만 fit)
k1 = (
    pd.to_numeric(data["업력"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .reset_index(drop=True)
    .to_frame("업력")
)

k1_pipe = Pipeline([
    ("clip", QuantileClipper(lower=LOWER_QUANTILE, upper=UPPER_QUANTILE)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])
k1_pipe.fit(k1.iloc[train_idx])

def build16(processed, k1_part):
    out = processed[[col_map[c] for c in feat15]].copy()
    out.columns = feat15
    out["업력"] = k1_pipe.transform(k1_part).ravel()
    return out[FEATURES_16]

X16_train = build16(X_train_processed, k1.iloc[train_idx])
X16_valid = build16(X_valid_processed, k1.iloc[valid_idx])
X16_test  = build16(X_test_processed,  k1.iloc[test_idx])

# 3) df16: 전체를 하나로 (원래 행 순서, 타깃 + 식별/연도 포함)
df16 = pd.concat([
    X16_train.assign(**{TARGET_COL: y_train}),
    X16_valid.assign(**{TARGET_COL: y_valid}),
    X16_test.assign(**{TARGET_COL: y_test}),
]).sort_index()

df16.insert(0, ID_COL, metadata.loc[df16.index, ID_COL].values)
df16["결산연도"] = metadata.loc[df16.index, "결산연도"].values

print(df16.shape)        # (3140, 19)
print("결측/무한대:", int(df16[FEATURES_16].isna().sum().sum()))
df16.head()

#### 데이터 분할

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

from xgboost import XGBClassifier

# ============================================================
# df16 분할 (148개 버전과 동일한 분할 방식)
# ============================================================
SEED = 42

# 148개 버전의 분할을 보관 (동일성 검증용)
ref_train_idx, ref_valid_idx, ref_test_idx = (
    train_idx.copy(), valid_idx.copy(), test_idx.copy()
)

assert df16.index.equals(X.index), "df16 행 순서가 원본과 다릅니다."

X16 = df16[FEATURES_16].copy()
y16 = df16[TARGET_COL].astype(int).copy()
meta16 = df16[[ID_COL, "결산연도"]].copy()
industry16 = X.loc[df16.index, "산업코드"]     # 분포 확인용 (모델 입력 아님)

assert len(X16) == len(y16) == len(meta16)
assert set(y16.unique()) == {0, 1}

indices = np.arange(len(X16))

train_idx, temp_idx = train_test_split(
    indices, test_size=0.40, random_state=SEED,
    shuffle=True, stratify=y16,
)

valid_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED,
    shuffle=True, stratify=y16.iloc[temp_idx],
)

# 148개 버전과 행이 완전히 같은지 검증
assert np.array_equal(train_idx, ref_train_idx)
assert np.array_equal(valid_idx, ref_valid_idx)
assert np.array_equal(test_idx, ref_test_idx)
print("148개 버전과 동일한 분할 확인")

X_train, X_valid, X_test = (X16.iloc[i].copy() for i in (train_idx, valid_idx, test_idx))
y_train, y_valid, y_test = (y16.iloc[i].copy() for i in (train_idx, valid_idx, test_idx))
meta_train, meta_valid, meta_test = (meta16.iloc[i].copy() for i in (train_idx, valid_idx, test_idx))

assert set(train_idx).isdisjoint(valid_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(valid_idx).isdisjoint(test_idx)
assert len(train_idx) + len(valid_idx) + len(test_idx) == len(X16)

for left, right in [(meta_train, meta_valid), (meta_train, meta_test), (meta_valid, meta_test)]:
    assert set(left[ID_COL]).isdisjoint(set(right[ID_COL])), "같은 기업이 여러 세트에 포함됐습니다."

split_summary = pd.DataFrame({
    "데이터": ["train", "valid", "test"],
    "행수": [len(y_train), len(y_valid), len(y_test)],
    "원본코드1_개수": [y_train.eq(0).sum(), y_valid.eq(0).sum(), y_test.eq(0).sum()],
    "원본코드2_개수": [y_train.eq(1).sum(), y_valid.eq(1).sum(), y_test.eq(1).sum()],
    "원본코드2_비율": [y_train.mean(), y_valid.mean(), y_test.mean()],
})
print("\n[분할 결과]")
print(split_summary.to_string(index=False))

industry_distribution = pd.DataFrame({
    "train": industry16.iloc[train_idx].value_counts(normalize=True),
    "valid": industry16.iloc[valid_idx].value_counts(normalize=True),
    "test":  industry16.iloc[test_idx].value_counts(normalize=True),
}).fillna(0).sort_index()
print("\n[산업별 구성비]")
print(industry_distribution.round(3))

year_distribution = pd.DataFrame({
    "train": meta_train["결산연도"].value_counts(normalize=True, dropna=False),
    "valid": meta_valid["결산연도"].value_counts(normalize=True, dropna=False),
    "test":  meta_test["결산연도"].value_counts(normalize=True, dropna=False),
}).fillna(0).sort_index()
print("\n[결산연도별 구성비]")
print(year_distribution.round(3))

## 6. QSVM-KTA (커널-타깃 정렬)

### 구성

기존 QSVM은 각도 스케일 γ를 **검증셋 AUC**로 골랐다. 여기서는 **KTA(Kernel-Target Alignment, 커널-타깃 정렬)** 로 양자 커널의 설정을 고른다.

| 개념 | 설명 |
|---|---|
| KTA | 커널 행렬이 "같은 클래스끼리 비슷하고 다른 클래스끼리 다른" 정답 구조(y yᵀ)와 얼마나 닮았는지를 나타내는 값 (중심화 KTA, −1 ~ 1, 클수록 좋음) |
| 장점 | 훈련셋의 라벨만으로 계산하고 SVM을 학습하지 않아도 되므로, 검증셋을 설정 선택에 쓰지 않는다 |
| 한계 | KTA가 높다고 분류 성능이 반드시 높은 것은 아니다. 최종 판단은 검증·테스트 성능으로 한다 |

| 모델 | 구성 |
|---|---|
| A. QSVM-KTA (특징 맵 선택) | γ(각도 스케일)와 reps(특징 맵 반복 횟수 1~3)의 조합 중 훈련 부분표본의 KTA가 가장 높은 것을 선택 |
| B. QSVM-KTA (+특징별 가중치) | A에서 출발해, 입력 특징 i의 회전각에 곱하는 가중치 wᵢ를 KTA가 오르는 방향으로 학습 (특징을 하나씩 키우거나 줄여 보고 오르면 채택, 2회 탐색) |

- 양자 특징 맵은 기존과 같다: IQP/ZZ 계열, 인접 큐비트 ZZ 얽힘, 충실도 커널, 상태벡터를 NumPy로 정확히 계산.
- KTA 계산에는 훈련셋에서 클래스 비율을 맞춰 뽑은 부분표본을 쓴다 (8큐비트 400개, 16큐비트는 계산량 때문에 300개). 최종 SVM 학습은 훈련셋 전체로 한다.
- 분류기는 SVC(precomputed kernel)이고, **C만** 검증셋 ROC-AUC로 고른다. 테스트셋은 최종 평가에만 쓴다.
- 가중치 학습은 부분표본에 과적합될 수 있으므로, 검증셋 KTA와 검증·테스트 성능으로 확인한다.
- 8큐비트는 변수 16개를 PCA 8개로 줄여 입력하고, 16큐비트는 PCA 없이 변수 1개를 큐비트 1개에 그대로 넣는다. 따라서 16큐비트의 특징별 가중치는 변수별 가중치로 읽을 수 있다.
- 업력(K1)이 포함된 16개 변수이므로, 업력의 산출 시점에 대한 데이터 누수 가능성은 기존 16개 버전과 동일하게 남아 있다.
- 16큐비트는 상태벡터가 65,536차원이라 커널 한 번에 수 초가 걸리고, 최종 학습은 모델당 1분 안팎이다.

#### 양자 커널 함수

In [ ]:
# ============================================================
# QSVM: NumPy 상태벡터 시뮬레이션 (추가 설치 불필요)
# ============================================================

import time

import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    classification_report,
    confusion_matrix,
)


# ------------------------------------------------------------
# 1. 양자 특징 맵 (IQP/ZZ 계열, 인접 큐비트 얽힘)
#
#   |psi(x)> = D(x) H^n D(x) H^n |0>
#   D(x) = exp( i [ sum_i x_i z_i + sum_i x_i x_{i+1} z_i z_{i+1} ] ),  z = +-1
# ------------------------------------------------------------

def _zmatrix(n):
    index = np.arange(2 ** n)[:, None]
    bits = (index >> np.arange(n - 1, -1, -1)) & 1

    return (1 - 2 * bits).astype(np.float32)


def _hadamard_all(state, n):
    batch = state.shape[0]

    for k in range(n):
        block = state.reshape(batch, 2 ** k, 2, 2 ** (n - k - 1))

        plus = block[:, :, 0] + block[:, :, 1]
        minus = block[:, :, 0] - block[:, :, 1]

        state = (
            np.stack((plus, minus), axis=2).reshape(batch, -1)
            * np.float32(1 / np.sqrt(2))
        )

    return state


def feature_states(features, reps=2, batch=128):
    features = np.asarray(features, dtype=np.float32)
    n = features.shape[1]

    z = _zmatrix(n)
    z_pair = z[:, :-1] * z[:, 1:]

    states = np.empty((len(features), 2 ** n), dtype=np.complex64)

    for start in range(0, len(features), batch):
        x = features[start:start + batch]

        phase = x @ z.T + (x[:, :-1] * x[:, 1:]) @ z_pair.T
        diagonal = np.exp(1j * phase).astype(np.complex64, copy=False)

        state = np.full(
            (len(x), 2 ** n),
            1 / np.sqrt(2 ** n),
            dtype=np.complex64,
        )

        for r in range(reps):
            state = state * diagonal

            if r < reps - 1:
                state = _hadamard_all(state, n)

        states[start:start + batch] = state

    return states


def fidelity_kernel(states_a, states_b_conj, chunk=256):
    """K = abs(<psi_a|psi_b>)^2"""

    kernel = np.empty(
        (len(states_a), len(states_b_conj)),
        dtype=np.float64,
    )

    for start in range(0, len(states_a), chunk):
        overlap = states_a[start:start + chunk] @ states_b_conj.T
        kernel[start:start + chunk] = np.abs(overlap) ** 2

    return kernel


# ------------------------------------------------------------
# 2. 성능 지표
# ------------------------------------------------------------

def qsvm_metrics(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels).reshape(-1)
    predictions = (probabilities >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(labels, predictions),
        "Balanced accuracy": balanced_accuracy_score(labels, predictions),
        "Precision_code2": precision_score(
            labels, predictions, zero_division=0
        ),
        "Recall_code2": recall_score(
            labels, predictions, zero_division=0
        ),
        "F1_code2": f1_score(labels, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(labels, probabilities),
        "Average precision": average_precision_score(
            labels, probabilities
        ),
        "Log loss": log_loss(labels, probabilities, labels=[0, 1]),
    }


C_GRID = [0.1, 1, 10, 100, 1000]


# ------------------------------------------------------------
# 3. QSVM 학습: 각도 스케일(gamma)과 C를 검증셋 AUC로 선택
# ------------------------------------------------------------

def run_qsvm(
    angles_train, angles_valid, angles_test,
    labels_train, labels_valid, labels_test,
    gammas, reps=2,
):
    best = None

    for gamma in gammas:
        started = time.time()

        states_train = feature_states(gamma * angles_train, reps)
        states_train_conj = states_train.conj()

        kernel_train = fidelity_kernel(states_train, states_train_conj)
        kernel_valid = fidelity_kernel(
            feature_states(gamma * angles_valid, reps),
            states_train_conj,
        )

        size = len(kernel_train)
        off_diagonal = (
            (kernel_train.sum() - np.trace(kernel_train))
            / (size * size - size)
        )

        for c in C_GRID:
            svm = SVC(kernel="precomputed", C=c).fit(
                kernel_train, labels_train
            )

            auc = roc_auc_score(
                labels_valid, svm.decision_function(kernel_valid)
            )

            print(
                f"  gamma={gamma:<5} C={c:<5} "
                f"valid AUC={auc:.4f}  "
                f"(커널 비대각 평균={off_diagonal:.4f})"
            )

            if best is None or auc > best["auc"]:
                best = {
                    "auc": auc,
                    "gamma": gamma,
                    "C": c,
                    "kernel_train": kernel_train,
                    "kernel_valid": kernel_valid,
                    "states_train_conj": states_train_conj,
                }

        print(f"  [gamma={gamma}] {time.time() - started:.0f}초")

    gamma, c = best["gamma"], best["C"]

    if len(gammas) > 1 and gamma in (gammas[0], gammas[-1]):
        print(
            "\n※ 선택된 gamma가 탐색 범위의 끝입니다. "
            "범위를 넓혀 다시 확인하세요."
        )

    if c in (C_GRID[0], C_GRID[-1]):
        print(
            "※ 선택된 C가 탐색 범위의 끝입니다. "
            "C_GRID를 넓혀 다시 확인하세요."
        )

    kernel_test = fidelity_kernel(
        feature_states(gamma * angles_test, reps),
        best["states_train_conj"],
    )

    svm = SVC(
        kernel="precomputed",
        C=c,
        probability=True,
        random_state=SEED,
    ).fit(best["kernel_train"], labels_train)

    probabilities = {
        "train": svm.predict_proba(best["kernel_train"])[:, 1],
        "valid": svm.predict_proba(best["kernel_valid"])[:, 1],
        "test": svm.predict_proba(kernel_test)[:, 1],
    }

    labels = {
        "train": labels_train,
        "valid": labels_valid,
        "test": labels_test,
    }

    results = pd.DataFrame({
        split: qsvm_metrics(labels[split], probabilities[split])
        for split in probabilities
    }).T

    print(f"\n선택된 설정: gamma={gamma}, C={c}")

    return results, {
        "gamma": gamma,
        "C": c,
        "probabilities": probabilities,
    }


# ------------------------------------------------------------
# 4. 고전 기준선: 같은 입력에 RBF-SVM / 로지스틱 회귀
# ------------------------------------------------------------

def run_classical(
    angles_train, angles_valid, angles_test,
    labels_train, labels_valid, labels_test,
):
    best = None

    for gamma in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]:
        for c in C_GRID:
            model = SVC(kernel="rbf", gamma=gamma, C=c).fit(
                angles_train, labels_train
            )

            auc = roc_auc_score(
                labels_valid, model.decision_function(angles_valid)
            )

            if best is None or auc > best[0]:
                best = (auc, gamma, c)

    _, gamma, c = best

    rbf = SVC(
        kernel="rbf",
        gamma=gamma,
        C=c,
        probability=True,
        random_state=SEED,
    ).fit(angles_train, labels_train)

    logistic = LogisticRegression(max_iter=2000).fit(
        angles_train, labels_train
    )

    return pd.DataFrame({
        f"RBF-SVM (gamma={gamma}, C={c})": qsvm_metrics(
            labels_test, rbf.predict_proba(angles_test)[:, 1]
        ),
        "로지스틱 회귀": qsvm_metrics(
            labels_test, logistic.predict_proba(angles_test)[:, 1]
        ),
    }).T

#### KTA 함수

In [ ]:
# ============================================================
# KTA (Kernel-Target Alignment, 커널-타깃 정렬)
#
#   KTA(K, y) = <Kc, y yᵀ>_F / (||Kc||_F · ||y yᵀ||_F)
#
#   Kc : 중심화한 커널 행렬,  y : 라벨(+1 / -1)을 평균 0으로 맞춘 벡터
#
# 커널 행렬이 "같은 클래스끼리 비슷하고 다른 클래스끼리 다르다"는
# 정답 구조(y yᵀ)와 얼마나 닮았는지를 -1~1로 나타낸다.
# 훈련셋의 라벨만 쓰고, 모델(SVM)을 학습하지 않고도 계산할 수 있다.
# ============================================================

def centered_kta(kernel, labels):
    y = 2.0 * np.asarray(labels, dtype=np.float64).reshape(-1) - 1.0
    y = y - y.mean()

    kc = (
        kernel
        - kernel.mean(axis=0, keepdims=True)
        - kernel.mean(axis=1, keepdims=True)
        + kernel.mean()
    )

    numerator = y @ kc @ y
    denominator = np.linalg.norm(kc) * (y @ y)

    return float(numerator / denominator)


def make_subsample(angles, labels, size, seed=42):
    """훈련셋에서 클래스 비율을 맞춰 size개를 뽑는다 (KTA 계산용)."""

    labels = np.asarray(labels).reshape(-1)
    rng = np.random.default_rng(seed)

    chosen = []

    for value in (0, 1):
        index = np.flatnonzero(labels == value)
        take = min(size // 2, len(index))
        chosen.append(rng.choice(index, size=take, replace=False))

    chosen = np.sort(np.concatenate(chosen))

    return angles[chosen], labels[chosen]


def kernel_on(angles, scale, reps):
    """angles * scale 을 특징 맵에 넣어 만든 충실도 커널 (샘플끼리)"""

    states = feature_states(angles * scale, reps)

    return fidelity_kernel(states, states.conj())


# ------------------------------------------------------------
# 1단계: 특징 맵(gamma, reps)을 KTA로 선택
# ------------------------------------------------------------

def kta_select_feature_map(angles, labels, gammas, reps_list, size):
    sub_angles, sub_labels = make_subsample(angles, labels, size)

    rows = []

    for reps in reps_list:
        for gamma in gammas:
            started = time.time()

            score = centered_kta(
                kernel_on(sub_angles, gamma, reps), sub_labels
            )

            rows.append({"reps": reps, "gamma": gamma, "KTA": score})

            print(
                f"  reps={reps} gamma={gamma:<6} KTA={score:.4f} "
                f"({time.time() - started:.1f}초)"
            )

    table = pd.DataFrame(rows)
    best = table.loc[table["KTA"].idxmax()]

    return table, best


# ------------------------------------------------------------
# 2단계: 특징별 가중치를 KTA로 학습
#
# 입력 특징 i의 회전각을 w_i 배 한다 (w_i 가 클수록 그 특징을 크게 반영).
# 미분 없이, 특징을 하나씩 w_i 를 키우거나 줄여 보고 KTA가 오르면 채택한다.
# ------------------------------------------------------------

def kta_learn_weights(
    angles, labels, gamma, reps, size,
    factor_schedule=((0.5, 2.0), (0.75, 1.33)),
    bounds=(0.1, 5.0),
):
    sub_angles, sub_labels = make_subsample(angles, labels, size)

    n_features = angles.shape[1]
    weights = np.ones(n_features)

    def score_of(w):
        return centered_kta(
            kernel_on(sub_angles, gamma * w, reps), sub_labels
        )

    best = score_of(weights)
    trace = [("시작", best)]

    print(f"  시작 KTA = {best:.4f}")

    for sweep, factors in enumerate(factor_schedule, start=1):
        started = time.time()

        for i in range(n_features):
            for factor in factors:
                candidate = weights.copy()
                candidate[i] = np.clip(
                    weights[i] * factor, bounds[0], bounds[1]
                )

                if candidate[i] == weights[i]:
                    continue

                score = score_of(candidate)

                if score > best + 1e-6:
                    best = score
                    weights = candidate

        trace.append((f"{sweep}회 탐색 후", best))

        print(
            f"  {sweep}회 탐색 후 KTA = {best:.4f} "
            f"({time.time() - started:.0f}초)"
        )

    return weights, trace


def alignment_on(angles, labels, gamma, weights, reps):
    """주어진 데이터 전체의 KTA (검증셋 점검용)"""

    return centered_kta(
        kernel_on(angles, gamma * weights, reps), labels
    )


labels_train = y_train.values
labels_valid = y_valid.values
labels_test = y_test.values

### qsvm-kta(8qubit)

#### 회로 세팅 (PCA 8개 → 큐비트 8개)

In [ ]:
# 16개 변수를 입력으로 사용 (이미 극단값 제한 / 대치 / RobustScaler 완료)
X_train_xgb = X_train[FEATURES_16]
X_valid_xgb = X_valid[FEATURES_16]
X_test_xgb = X_test[FEATURES_16]

print("입력 변수 수:", X_train_xgb.shape[1])

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ============================================================
# 1. 설정
# ============================================================

N_QUBITS = 8
SEED = 42

# 값이 클수록 tanh 포화가 완만해짐
# 테스트셋을 보고 조정하지 말고, 필요하면 검증셋에서 선택
ANGLE_SCALE = 2.0


# ============================================================
# 2. 입력 확인
# ============================================================

train_array = np.asarray(X_train_xgb, dtype=np.float64)
valid_array = np.asarray(X_valid_xgb, dtype=np.float64)
test_array = np.asarray(X_test_xgb, dtype=np.float64)

for name, array in [
    ("train", train_array),
    ("valid", valid_array),
    ("test", test_array),
]:
    if array.ndim != 2:
        raise ValueError(f"{name}: 2차원 배열이어야 합니다.")

    if not np.isfinite(array).all():
        raise ValueError(f"{name}: 결측 또는 무한대가 있습니다.")

if not (
    train_array.shape[1]
    == valid_array.shape[1]
    == test_array.shape[1]
):
    raise ValueError("훈련·검증·테스트의 변수 수가 다릅니다.")


# ============================================================
# 3. 훈련셋에서 상수 변수 제거
#
# 예: 항상 0인 결측 표시 변수
# ============================================================

q_constant_filter = VarianceThreshold(threshold=0.0)

train_selected = q_constant_filter.fit_transform(train_array)
valid_selected = q_constant_filter.transform(valid_array)
test_selected = q_constant_filter.transform(test_array)

max_components = min(
    train_selected.shape[0] - 1,
    train_selected.shape[1],
)

if not 1 <= N_QUBITS <= max_components:
    raise ValueError(
        f"N_QUBITS는 1~{max_components} 사이여야 합니다."
    )


# ============================================================
# 4. PCA 전 표준화
#
# 기존 RobustScaler가 잘못됐다는 뜻이 아님.
# PCA에서 변수별 분산 차이의 영향을 조절하기 위한 추가 단계.
#
# 원핫/결측 표시처럼 희소한 이진변수도 확대될 수 있으므로
# 이후 표준화 유무를 검증셋에서 비교할 수 있음.
# ============================================================

q_input_scaler = StandardScaler()

train_scaled = q_input_scaler.fit_transform(train_selected)
valid_scaled = q_input_scaler.transform(valid_selected)
test_scaled = q_input_scaler.transform(test_selected)


# ============================================================
# 5. PCA로 소수 특징에 압축
#
# y를 사용하지 않는 비지도 차원 축소
# ============================================================

q_pca = PCA(
    n_components=N_QUBITS,
    svd_solver="full",
    random_state=SEED,
)

Z_train = q_pca.fit_transform(train_scaled)
Z_valid = q_pca.transform(valid_scaled)
Z_test = q_pca.transform(test_scaled)

print("성분별 설명 분산 비율:")
print(np.round(q_pca.explained_variance_ratio_, 4))

print(
    "누적 설명 분산 비율:",
    round(q_pca.explained_variance_ratio_.sum(), 4),
)

# 설명 분산 비율이 높다고 분류 정보가 반드시 잘 보존되는 것은 아님.
# 반대로 낮더라도 분류에 필요한 정보 일부를 담을 수 있으므로
# 최종 판단은 검증 분류 성능으로 해야 함.


# ============================================================
# 6. PCA 성분을 훈련셋 기준으로 표준화
#
# 각 성분의 입력각 크기가 지나치게 달라지는 것을 방지
# ============================================================

q_component_scaler = StandardScaler()

Z_train_standard = q_component_scaler.fit_transform(Z_train)
Z_valid_standard = q_component_scaler.transform(Z_valid)
Z_test_standard = q_component_scaler.transform(Z_test)


# ============================================================
# 7. 양자회로 입력각으로 변환
#
# theta = pi * tanh(z / ANGLE_SCALE)
#
# 단순 선형 각도 변환에서 큰 값이 반복적인 회전으로
# 겹치는 문제를 줄이기 위한 유계 변환.
# 단, 극단 영역의 차이가 압축되는 단점은 있음.
# ============================================================

def to_quantum_angles(z, scale=ANGLE_SCALE):
    if scale <= 0:
        raise ValueError("scale은 양수여야 합니다.")

    return np.pi * np.tanh(
        np.asarray(z, dtype=np.float64) / scale
    )


X_train_q = to_quantum_angles(Z_train_standard)
X_valid_q = to_quantum_angles(Z_valid_standard)
X_test_q = to_quantum_angles(Z_test_standard)


# ============================================================
# 8. 데이터프레임으로 정리
# ============================================================

angle_columns = [
    f"theta_{i}"
    for i in range(N_QUBITS)
]

X_train_q_df = pd.DataFrame(
    X_train_q,
    index=X_train.index,
    columns=angle_columns,
)

X_valid_q_df = pd.DataFrame(
    X_valid_q,
    index=X_valid.index,
    columns=angle_columns,
)

X_test_q_df = pd.DataFrame(
    X_test_q,
    index=X_test.index,
    columns=angle_columns,
)


# ============================================================
# 9. 결과 검증
# ============================================================

for name, array in [
    ("train", X_train_q),
    ("valid", X_valid_q),
    ("test", X_test_q),
]:
    assert np.isfinite(array).all()
    assert array.shape[1] == N_QUBITS
    assert np.abs(array).max() <= np.pi

    print(
        f"{name}: shape={array.shape}, "
        f"각도 범위=({array.min():.4f}, {array.max():.4f})"
    )

# 타깃은 기존 것을 그대로 사용
assert len(X_train_q) == len(y_train)
assert len(X_valid_q) == len(y_valid)
assert len(X_test_q) == len(y_test)

print("\n양자회로 입력 예시:")
print(X_train_q_df.head())

#### 모델 A. KTA로 특징 맵 선택

In [ ]:
# ============================================================
# 8큐비트: KTA로 특징 맵(gamma, reps) 선택 → 모델 A
# ============================================================

GAMMAS_KTA_8Q = (0.02, 0.03, 0.05, 0.075, 0.1, 0.15, 0.25, 0.4)
REPS_KTA = (1, 2, 3)

# KTA 계산에 쓰는 훈련셋 부분표본 크기 (커널 행렬이 400 x 400)
N_KTA_8Q = 400

kta8_labels = [f"PC{i + 1}" for i in range(X_train_q.shape[1])]

print("[KTA로 특징 맵(gamma, reps) 선택: 훈련셋 부분표본]")

kta8_table, kta8_best = kta_select_feature_map(
    X_train_q, labels_train,
    GAMMAS_KTA_8Q, REPS_KTA, N_KTA_8Q,
)

print("\n[KTA 표 (행: gamma, 열: reps)]")
print(
    kta8_table
    .pivot(index="gamma", columns="reps", values="KTA")
    .round(4)
    .to_string()
)

kta8_gamma = float(kta8_best["gamma"])
kta8_reps = int(kta8_best["reps"])

print(
    f"\n선택된 특징 맵: gamma={kta8_gamma}, reps={kta8_reps}, "
    f"KTA={kta8_best['KTA']:.4f}"
)

if kta8_gamma in (GAMMAS_KTA_8Q[0], GAMMAS_KTA_8Q[-1]):
    print("※ 선택된 gamma가 탐색 범위의 끝입니다. 범위를 넓혀 다시 확인하세요.")

print("\n[QSVM-KTA(특징 맵 선택) 훈련: C는 검증셋 AUC로 선택]")

kta8_gamma_results, kta8_gamma_info = run_qsvm(
    X_train_q, X_valid_q, X_test_q,
    labels_train, labels_valid, labels_test,
    gammas=(kta8_gamma,),
    reps=kta8_reps,
)

print("\n[QSVM-KTA(특징 맵 선택) 8큐비트 성능]")
print(kta8_gamma_results.round(4).to_string())

kta8_gamma_pred = (
    kta8_gamma_info["probabilities"]["test"] >= 0.5
).astype(int)

print("\n[테스트 혼동행렬]")
print(
    pd.DataFrame(
        confusion_matrix(labels_test, kta8_gamma_pred, labels=[0, 1]),
        index=["실제 원본코드1", "실제 원본코드2"],
        columns=["예측 원본코드1", "예측 원본코드2"],
    )
)

#### 모델 B. KTA로 특징별 가중치 학습

In [ ]:
# ============================================================
# 8큐비트: KTA로 특징별 가중치 학습 → 모델 B
#   (모델 A에서 고른 gamma, reps에서 출발)
# ============================================================

print("[KTA로 특징별 가중치 학습: 훈련셋 부분표본]")

kta8_weights, kta8_trace = kta_learn_weights(
    X_train_q, labels_train,
    kta8_gamma, kta8_reps, N_KTA_8Q,
)

weight_table = (
    pd.DataFrame({
        "특징": kta8_labels,
        "가중치": kta8_weights,
    })
    .sort_values("가중치", ascending=False)
    .reset_index(drop=True)
)

print("\n[학습된 특징별 가중치 (1.0 = 변화 없음, 범위 0.1~5)]")
print(weight_table.round(3).to_string())

# 가중치가 훈련 부분표본에만 맞춰졌는지 검증셋 KTA로 점검
ones = np.ones(X_train_q.shape[1])

kta8_valid_kta_before = alignment_on(
    X_valid_q, labels_valid, kta8_gamma, ones, kta8_reps
)
kta8_valid_kta_after = alignment_on(
    X_valid_q, labels_valid, kta8_gamma, kta8_weights, kta8_reps
)

print(
    f"\n검증셋 KTA: 가중치 학습 전 {kta8_valid_kta_before:.4f} "
    f"→ 후 {kta8_valid_kta_after:.4f}"
)

print("\n[QSVM-KTA(+특징별 가중치) 훈련: C는 검증셋 AUC로 선택]")

kta8_weight_results, kta8_weight_info = run_qsvm(
    X_train_q * kta8_weights,
    X_valid_q * kta8_weights,
    X_test_q * kta8_weights,
    labels_train, labels_valid, labels_test,
    gammas=(kta8_gamma,),
    reps=kta8_reps,
)

print("\n[QSVM-KTA(+특징별 가중치) 8큐비트 성능]")
print(kta8_weight_results.round(4).to_string())

kta8_weight_pred = (
    kta8_weight_info["probabilities"]["test"] >= 0.5
).astype(int)

print("\n[테스트 혼동행렬]")
print(
    pd.DataFrame(
        confusion_matrix(labels_test, kta8_weight_pred, labels=[0, 1]),
        index=["실제 원본코드1", "실제 원본코드2"],
        columns=["예측 원본코드1", "예측 원본코드2"],
    )
)

### qsvm-kta(16qubit)

#### 회로 세팅 (PCA 없이 변수 16개 → 큐비트 16개)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. 설정
#
# 독립변수 16개 → 큐비트 16개 (변수 1개당 큐비트 1개)
# PCA는 쓰지 않는다. 변수 자체를 회전각으로 인코딩한다.
# ============================================================

N_QUBITS_16 = 16
SEED_16 = 42

# 8큐비트 세팅과 같은 값
ANGLE_SCALE_16 = 2.0


# ============================================================
# 2. 입력 확인
#
# 이미 극단값 제한 / 대치 / RobustScaler가 끝난 16개 변수
# ============================================================

train_array16 = np.asarray(X_train[FEATURES_16], dtype=np.float64)
valid_array16 = np.asarray(X_valid[FEATURES_16], dtype=np.float64)
test_array16 = np.asarray(X_test[FEATURES_16], dtype=np.float64)

for name, array in [
    ("train", train_array16),
    ("valid", valid_array16),
    ("test", test_array16),
]:
    if array.ndim != 2:
        raise ValueError(f"{name}: 2차원 배열이어야 합니다.")

    if not np.isfinite(array).all():
        raise ValueError(f"{name}: 결측 또는 무한대가 있습니다.")


# ============================================================
# 3. 훈련셋에서 상수 변수 제거
#
# 16개가 모두 남아야 큐비트 16개와 1:1로 대응된다.
# ============================================================

q16_constant_filter = VarianceThreshold(threshold=0.0)

train_selected16 = q16_constant_filter.fit_transform(train_array16)
valid_selected16 = q16_constant_filter.transform(valid_array16)
test_selected16 = q16_constant_filter.transform(test_array16)

kept_features16 = [
    feature
    for feature, keep in zip(
        FEATURES_16, q16_constant_filter.get_support()
    )
    if keep
]

if train_selected16.shape[1] != N_QUBITS_16:
    removed = sorted(set(FEATURES_16) - set(kept_features16))
    raise ValueError(
        f"상수 변수가 제거되어 {train_selected16.shape[1]}개만 남았습니다. "
        f"제거된 변수: {removed}"
    )


# ============================================================
# 4. 표준화 (훈련셋 기준)
#
# 변수마다 입력각의 크기가 달라지지 않도록 평균 0, 표준편차 1로 맞춘다.
# ============================================================

q16_input_scaler = StandardScaler()

train_scaled16 = q16_input_scaler.fit_transform(train_selected16)
valid_scaled16 = q16_input_scaler.transform(valid_selected16)
test_scaled16 = q16_input_scaler.transform(test_selected16)


# ============================================================
# 5. 양자회로 입력각으로 변환
#
# theta = pi * tanh(z / ANGLE_SCALE)
# ============================================================

def to_quantum_angles16(z, scale=ANGLE_SCALE_16):
    if scale <= 0:
        raise ValueError("scale은 양수여야 합니다.")

    return np.pi * np.tanh(
        np.asarray(z, dtype=np.float64) / scale
    )


X_train_q16 = to_quantum_angles16(train_scaled16)
X_valid_q16 = to_quantum_angles16(valid_scaled16)
X_test_q16 = to_quantum_angles16(test_scaled16)


# ============================================================
# 6. 데이터프레임으로 정리 (큐비트 i = i번째 변수)
# ============================================================

angle_columns16 = [f"theta_{feature}" for feature in kept_features16]

X_train_q16_df = pd.DataFrame(
    X_train_q16, index=X_train.index, columns=angle_columns16
)
X_valid_q16_df = pd.DataFrame(
    X_valid_q16, index=X_valid.index, columns=angle_columns16
)
X_test_q16_df = pd.DataFrame(
    X_test_q16, index=X_test.index, columns=angle_columns16
)


# ============================================================
# 7. 결과 검증
# ============================================================

for name, array in [
    ("train", X_train_q16),
    ("valid", X_valid_q16),
    ("test", X_test_q16),
]:
    assert np.isfinite(array).all()
    assert array.shape[1] == N_QUBITS_16
    assert np.abs(array).max() <= np.pi

    print(
        f"{name}: shape={array.shape}, "
        f"각도 범위=({array.min():.4f}, {array.max():.4f})"
    )

assert len(X_train_q16) == len(y_train)
assert len(X_valid_q16) == len(y_valid)
assert len(X_test_q16) == len(y_test)

print("\n큐비트 ↔ 변수 대응:")
for qubit, feature in enumerate(kept_features16):
    print(f"  qubit {qubit:2d} : {feature}")

print("\n양자회로 입력 예시:")
print(X_train_q16_df.head())

#### 모델 A. KTA로 특징 맵 선택

In [ ]:
# ============================================================
# 16큐비트: KTA로 특징 맵(gamma, reps) 선택 → 모델 A
# ============================================================

GAMMAS_KTA_16Q = (0.01, 0.02, 0.03, 0.05, 0.1, 0.2)
REPS_KTA = (1, 2, 3)

# KTA 계산에 쓰는 훈련셋 부분표본 크기 (커널 행렬이 300 x 300)
N_KTA_16Q = 300

kta16_labels = list(kept_features16)

print("[KTA로 특징 맵(gamma, reps) 선택: 훈련셋 부분표본]")

kta16_table, kta16_best = kta_select_feature_map(
    X_train_q16, labels_train,
    GAMMAS_KTA_16Q, REPS_KTA, N_KTA_16Q,
)

print("\n[KTA 표 (행: gamma, 열: reps)]")
print(
    kta16_table
    .pivot(index="gamma", columns="reps", values="KTA")
    .round(4)
    .to_string()
)

kta16_gamma = float(kta16_best["gamma"])
kta16_reps = int(kta16_best["reps"])

print(
    f"\n선택된 특징 맵: gamma={kta16_gamma}, reps={kta16_reps}, "
    f"KTA={kta16_best['KTA']:.4f}"
)

if kta16_gamma in (GAMMAS_KTA_16Q[0], GAMMAS_KTA_16Q[-1]):
    print("※ 선택된 gamma가 탐색 범위의 끝입니다. 범위를 넓혀 다시 확인하세요.")

print("\n[QSVM-KTA(특징 맵 선택) 훈련: C는 검증셋 AUC로 선택]")

kta16_gamma_results, kta16_gamma_info = run_qsvm(
    X_train_q16, X_valid_q16, X_test_q16,
    labels_train, labels_valid, labels_test,
    gammas=(kta16_gamma,),
    reps=kta16_reps,
)

print("\n[QSVM-KTA(특징 맵 선택) 16큐비트 성능]")
print(kta16_gamma_results.round(4).to_string())

kta16_gamma_pred = (
    kta16_gamma_info["probabilities"]["test"] >= 0.5
).astype(int)

print("\n[테스트 혼동행렬]")
print(
    pd.DataFrame(
        confusion_matrix(labels_test, kta16_gamma_pred, labels=[0, 1]),
        index=["실제 원본코드1", "실제 원본코드2"],
        columns=["예측 원본코드1", "예측 원본코드2"],
    )
)

#### 모델 B. KTA로 특징별 가중치 학습

In [ ]:
# ============================================================
# 16큐비트: KTA로 특징별 가중치 학습 → 모델 B
#   (모델 A에서 고른 gamma, reps에서 출발)
# ============================================================

print("[KTA로 특징별 가중치 학습: 훈련셋 부분표본]")

kta16_weights, kta16_trace = kta_learn_weights(
    X_train_q16, labels_train,
    kta16_gamma, kta16_reps, N_KTA_16Q,
)

weight_table = (
    pd.DataFrame({
        "특징": kta16_labels,
        "가중치": kta16_weights,
    })
    .sort_values("가중치", ascending=False)
    .reset_index(drop=True)
)

print("\n[학습된 특징별 가중치 (1.0 = 변화 없음, 범위 0.1~5)]")
print(weight_table.round(3).to_string())

# 가중치가 훈련 부분표본에만 맞춰졌는지 검증셋 KTA로 점검
ones = np.ones(X_train_q16.shape[1])

kta16_valid_kta_before = alignment_on(
    X_valid_q16, labels_valid, kta16_gamma, ones, kta16_reps
)
kta16_valid_kta_after = alignment_on(
    X_valid_q16, labels_valid, kta16_gamma, kta16_weights, kta16_reps
)

print(
    f"\n검증셋 KTA: 가중치 학습 전 {kta16_valid_kta_before:.4f} "
    f"→ 후 {kta16_valid_kta_after:.4f}"
)

print("\n[QSVM-KTA(+특징별 가중치) 훈련: C는 검증셋 AUC로 선택]")

kta16_weight_results, kta16_weight_info = run_qsvm(
    X_train_q16 * kta16_weights,
    X_valid_q16 * kta16_weights,
    X_test_q16 * kta16_weights,
    labels_train, labels_valid, labels_test,
    gammas=(kta16_gamma,),
    reps=kta16_reps,
)

print("\n[QSVM-KTA(+특징별 가중치) 16큐비트 성능]")
print(kta16_weight_results.round(4).to_string())

kta16_weight_pred = (
    kta16_weight_info["probabilities"]["test"] >= 0.5
).astype(int)

print("\n[테스트 혼동행렬]")
print(
    pd.DataFrame(
        confusion_matrix(labels_test, kta16_weight_pred, labels=[0, 1]),
        index=["실제 원본코드1", "실제 원본코드2"],
        columns=["예측 원본코드1", "예측 원본코드2"],
    )
)

## 모델 성능 비교

In [ ]:
# ============================================================
# 테스트 성능 비교 (KTA 모델)
# ============================================================

rows = {
    "QSVM-KTA(8qubit, 특징 맵 선택)": kta8_gamma_results.loc["test"],
    "QSVM-KTA(8qubit, +특징별 가중치)": kta8_weight_results.loc["test"],
    "QSVM-KTA(16qubit, 특징 맵 선택)": kta16_gamma_results.loc["test"],
    "QSVM-KTA(16qubit, +특징별 가중치)": kta16_weight_results.loc["test"],
}

comparison = pd.DataFrame(rows).T

print("[테스트 성능 비교]")
print(comparison.round(4).to_string())


# 참고: 앞선 실험 결과 (테스트셋)

reference = pd.DataFrame(
    {
        "XGBoost(16개 변수)": [0.8615, 0.9453, 0.3044],
        "XGBoost(PCA 8개)": [0.8232, 0.8946, 0.4052],
        "QSVM(8qubit, 검증셋 AUC로 gamma 선택)": [0.8376, 0.8973, 0.3983],
        "QSVM(16qubit, 검증셋 AUC로 gamma 선택)": [0.8583, 0.9268, 0.3419],
        "VQC(8qubit)": [0.6990, 0.7791, 0.5619],
        "VQC(8qubit+drc)": [0.7532, 0.8313, 0.5128],
    },
    index=["Accuracy", "ROC-AUC", "Log loss"],
).T

print("\n[참고: 이전 실험 값]")
print(reference.to_string())

## 7. 연구보고서

### 7.1 목적
t-test와 로지스틱 회귀로 고른 독립변수 16개를 입력으로, QSVM의 설정을 KTA(커널-타깃 정렬)로 고르거나 학습하면 성능이 달라지는지 확인한다. 8큐비트(PCA 8개)와 16큐비트(변수 16개를 큐비트 16개에 직접 입력)를 각각 본다.

### 7.2 데이터와 입력
- 3,140개 기업(정상/폐업 1:1), 훈련/검증/테스트 = 1,884 / 628 / 628 (148개 버전과 동일한 분할임을 확인)
- 입력 변수 16개: 영업이익률, 재료비/매출액, 감가상각률, 금융비용대총부채비율, 금융비용대총비용비율, EBITDA/총차입금, 차입금의존도, 장단기대여금/총자산, 법인세부담률, 순운전자본대총자본비율, 유동자산구성비율, 유보액대총자산비율, 재고자산대유동자산비율, 매입채무회전기간, 자본금회전율(회), 업력
- 8큐비트: 16개 변수 → 표준화 → PCA 8개 → 표준화 → θ = π·tanh(z/2). PCA 누적 설명 분산 0.7462
- 16큐비트: PCA 없이 변수 1개를 큐비트 1개에 입력. 따라서 특징별 가중치를 변수별 가중치로 읽을 수 있다.

### 7.3 방법
- 양자 특징 맵: IQP/ZZ 계열, 인접 큐비트 ZZ 얽힘, 충실도 커널, 분류기는 SVC(precomputed)
- 모델 A (특징 맵 선택): γ와 reps(1~3) 조합 중 훈련 부분표본(8큐비트 400개, 16큐비트 300개)의 KTA가 최대인 조합 선택
- 모델 B (+특징별 가중치): A에서 출발해 특징별 각도 가중치를 KTA가 오르는 방향으로 2회 탐색
- C는 두 모델 모두 검증셋 ROC-AUC로 선택, 테스트셋은 최종 평가에만 사용

### 7.4 결과
**선택된 설정**

| 모델 | γ | reps | 부분표본 KTA (선택 → 가중치 학습 후) | 검증셋 KTA (학습 전 → 후) | C | 검증 AUC |
|---|---:|---:|---|---|---:|---:|
| 8큐비트 A | 0.15 | 2 | 0.1789 | - | 10 | 0.8534 |
| 8큐비트 B | 0.15 | 2 | 0.1789 → 0.2995 | 0.1504 → 0.2513 | 100 | 0.8511 |
| 16큐비트 A | 0.10 | 2 | 0.2943 | - | 10 | 0.8807 |
| 16큐비트 B | 0.10 | 2 | 0.2943 → 0.4383 | 0.2337 → 0.3442 | 10 | 0.8799 |

**테스트 성능**

| 모델 | Accuracy | Precision | Recall | F1 | ROC-AUC | Avg. precision | Log loss |
|---|---:|---:|---:|---:|---:|---:|---:|
| QSVM-KTA 8큐비트 A | 0.8424 | 0.8248 | 0.8694 | 0.8465 | 0.8980 | 0.8721 | 0.3978 |
| QSVM-KTA 8큐비트 B | 0.8408 | 0.8242 | 0.8662 | 0.8447 | 0.9001 | 0.8843 | 0.3961 |
| QSVM-KTA 16큐비트 A | 0.8583 | 0.8399 | 0.8854 | 0.8620 | 0.9268 | 0.9217 | 0.3419 |
| QSVM-KTA 16큐비트 B | 0.8503 | 0.8374 | 0.8694 | 0.8531 | 0.9238 | 0.9172 | 0.3520 |

(Balanced accuracy는 테스트셋이 314/314라서 Accuracy와 같다.)

**테스트 혼동행렬 (행: 실제, 열: 예측, 순서: 정상/폐업)**

| 모델 | 정상→정상 | 정상→폐업 | 폐업→정상 | 폐업→폐업 |
|---|---:|---:|---:|---:|
| 8큐비트 A | 256 | 58 | 41 | 273 |
| 8큐비트 B | 256 | 58 | 42 | 272 |
| 16큐비트 A | 261 | 53 | 36 | 278 |
| 16큐비트 B | 261 | 53 | 41 | 273 |

**기존 QSVM(검증셋 AUC로 γ 선택) 대비 차이 (KTA − 기존)**

| 모델 | ΔAccuracy | ΔROC-AUC | ΔLog loss |
|---|---:|---:|---:|
| 8큐비트 A | +0.0048 | +0.0007 | -0.0005 |
| 8큐비트 B | +0.0032 | +0.0028 | -0.0022 |
| 16큐비트 A | 0.0000 | 0.0000 | 0.0000 |
| 16큐비트 B | -0.0080 | -0.0030 | +0.0101 |

기존 값: 8큐비트 0.8376 / 0.8973 / 0.3983, 16큐비트 0.8583 / 0.9268 / 0.3419 (Accuracy / AUC / Log loss). 16큐비트 A는 기존과 수치가 완전히 같아서, 두 방법이 같은 설정을 골랐을 가능성이 높다.

**참고 모델 (이전 실험, 테스트)**

| 모델 | Accuracy | ROC-AUC | Log loss |
|---|---:|---:|---:|
| XGBoost(16개 변수) | 0.8615 | 0.9453 | 0.3044 |
| XGBoost(PCA 8개) | 0.8232 | 0.8946 | 0.4052 |
| VQC(8큐비트) | 0.6990 | 0.7791 | 0.5619 |
| VQC(8큐비트+drc) | 0.7532 | 0.8313 | 0.5128 |

### 7.5 해석
- KTA로 설정을 골라도 기존 QSVM과 성능이 사실상 같다. 차이는 Accuracy ±0.008, AUC ±0.003 이내이고, 테스트 AUC 표준오차(약 0.015)보다 작다.
- 특징별 가중치 학습(B)은 KTA를 가장 크게 올린 경우다. 부분표본 KTA는 0.1789 → 0.2995(8큐비트), 0.2943 → 0.4383(16큐비트)로 올랐고, 검증셋 KTA도 0.1504 → 0.2513, 0.2337 → 0.3442로 함께 올랐다. 그런데 검증 AUC와 테스트 성능은 오르지 않았다. 커널이 라벨 구조와 더 맞춰져도 SVM의 분류 성능은 그대로였다.
- 16큐비트 B의 가중치는 변수별로 읽을 수 있다. 감가상각률(2.0), 금융비용대총비용비율(1.5), 유보액대총자산비율(1.5)이 커졌고, 영업이익률·차입금의존도·EBITDA/총차입금·업력 등 9개 변수는 0.375로 줄었다. 좌표 탐색의 거친 배율이므로 통계적 변수 중요도로 해석하면 안 된다.
- 16큐비트가 8큐비트보다 높다(AUC 0.898~0.900 → 0.924~0.927). 16큐비트는 PCA 없이 변수를 그대로 쓰므로, PCA 8개로 압축하며 잃은 정보(누적 설명 분산 0.7462)가 되살아난 효과로 보인다.
- 16큐비트 KTA-QSVM(AUC 0.924~0.927)은 XGBoost(AUC 0.9453)보다 낮고, VQC 계열(0.78~0.83)보다 높다. Accuracy는 XGBoost(0.8615)와 비슷한 0.850~0.858이다.
- reps=3에서 γ=0.2로 키우면 KTA가 급락한다(0.1470). 특징 맵이 깊고 각도가 클수록 커널이 단위행렬에 가까워지는 현상으로 보인다.

### 7.6 한계
- 데이터 분할 1개, 시드 1개 결과이고, 모두 시뮬레이터(상태벡터) 기반이다.
- 설정 선택용 KTA는 훈련 부분표본(400개/300개)으로 계산했다. 가중치 학습은 부분표본에 과적합될 수 있다.
- 16큐비트 가중치 학습은 약 220초, 최종 학습은 모델당 약 95초가 걸렸다.
- 업력(K1)이 포함된 변수 집합이라, 업력 산출 시점에 따른 데이터 누수 가능성은 기존 16개 버전과 동일하게 남아 있다.